In [119]:
import pandas as pd
import plotly.express as px
from django.contrib.admin import display
from sqlalchemy               import (create_engine)
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os
from sklearn.model_selection  import train_test_split
from sklearn.ensemble         import RandomForestClassifier
from sklearn.metrics          import classification_report,roc_auc_score, confusion_matrix
from sklearn.preprocessing    import StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.naive_bayes      import BernoulliNB

In [77]:
inpatient_path = r"C:\Users\Νίκος\Documents\N'work\thesis\Git_Repository\medicare-data-mining\data\raw\2015_2025\inpatient.csv"

inpatient_df = pd.read_csv(inpatient_path, sep="|", low_memory=False)

print(inpatient_df.shape)
print(inpatient_df.columns[:20])

(58066, 197)
Index(['BENE_ID', 'CLM_ID', 'NCH_NEAR_LINE_REC_IDENT_CD', 'NCH_CLM_TYPE_CD',
       'CLM_FROM_DT', 'CLM_THRU_DT', 'NCH_WKLY_PROC_DT', 'FI_CLM_PROC_DT',
       'CLAIM_QUERY_CODE', 'PRVDR_NUM', 'CLM_FAC_TYPE_CD',
       'CLM_SRVC_CLSFCTN_TYPE_CD', 'CLM_FREQ_CD', 'FI_NUM',
       'CLM_MDCR_NON_PMT_RSN_CD', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT',
       'NCH_PRMRY_PYR_CD', 'FI_CLM_ACTN_CD', 'PRVDR_STATE_CD'],
      dtype='str')


In [79]:
inpatient_df.columns = (
    inpatient_df.columns
        .str.lower()
        .str.strip()
)
inpatient_df.columns[:20]

Index(['bene_id', 'clm_id', 'nch_near_line_rec_ident_cd', 'nch_clm_type_cd',
       'clm_from_dt', 'clm_thru_dt', 'nch_wkly_proc_dt', 'fi_clm_proc_dt',
       'claim_query_code', 'prvdr_num', 'clm_fac_type_cd',
       'clm_srvc_clsfctn_type_cd', 'clm_freq_cd', 'fi_num',
       'clm_mdcr_non_pmt_rsn_cd', 'clm_pmt_amt', 'nch_prmry_pyr_clm_pd_amt',
       'nch_prmry_pyr_cd', 'fi_clm_actn_cd', 'prvdr_state_cd'],
      dtype='str')

In [80]:
inpatient_df["bene_id"] = inpatient_df["bene_id"].astype(str)

In [82]:
inpatient_df["clm_from_dt"] = pd.to_datetime(
    inpatient_df["clm_from_dt"],
    errors="coerce"
)

inpatient_df["year"] = inpatient_df["clm_from_dt"].dt.year
inpatient_df["year"].value_counts().sort_index()

year
2015     4050
2016     4928
2017     5515
2018     6297
2019     6791
2020     8775
2021     9249
2022    10598
2023     1863
Name: count, dtype: int64

In [83]:
money_cols = [
    "clm_pmt_amt",
    "clm_tot_chrg_amt",
    "nch_bene_ip_ddctbl_amt",
    "nch_bene_pta_coinsrnc_lblty_am"
]

for col in money_cols:
    if col in inpatient_df.columns:
        inpatient_df[col] = pd.to_numeric(inpatient_df[col], errors="coerce")

In [84]:
annual_cost = (
    inpatient_df
        .groupby(["bene_id", "year"])[money_cols]
        .sum()
        .reset_index()
)

In [85]:
annual_cost = annual_cost.rename(columns={
    "clm_pmt_amt": "total_inpatient_paid",
    "clm_tot_chrg_amt": "total_inpatient_charge",
    "nch_bene_ip_ddctbl_amt": "total_deductible",
    "nch_bene_pta_coinsrnc_lblty_am": "total_coinsurance"
})

In [86]:
annual_cost.describe()

,year,total_inpatient_paid,total_inpatient_charge,total_deductible,total_coinsurance
count,11545.000000,1.154500e+04,1.154500e+04,11545.000000,1.154500e+04
mean,2018.989259,6.859437e+04,6.859437e+04,48.237660,9.530741e+03
std,2.343299,5.776398e+05,5.776398e+05,440.049702,8.080733e+04
min,2015.000000,6.244000e+01,6.244000e+01,0.000000,0.000000e+00
25%,2017.000000,1.650000e+02,1.650000e+02,0.000000,0.000000e+00
50%,2019.000000,5.859360e+03,5.859360e+03,0.000000,6.342000e+01
75%,2021.000000,2.388212e+04,2.388212e+04,0.000000,2.937280e+03
max,2023.000000,2.572090e+07,2.572090e+07,16818.880000,3.656424e+06


In [87]:
dx_cols = [c for c in inpatient_df.columns if "dgns" in c]

print(dx_cols[:10])

['admtg_dgns_cd', 'prncpal_dgns_cd', 'icd_dgns_cd1', 'icd_dgns_cd2', 'icd_dgns_cd3', 'icd_dgns_cd4', 'icd_dgns_cd5', 'icd_dgns_cd6', 'icd_dgns_cd7', 'icd_dgns_cd8']


In [88]:
inpatient_df[dx_cols] = (
    inpatient_df[dx_cols]
        .fillna("")
        .astype(str)
        .apply(lambda x: x.str.strip())
)

In [89]:
disease_map = {
    "diabetes": r"\bE1[0-4]",
    "ckd": r"\bN18",
    "chf": r"\bI50",
    "copd": r"\bJ44",
    "cancer": r"\bC",
    "stroke": r"\bI6[3-4]"
}

In [90]:
for disease, pattern in disease_map.items():
    inpatient_df[disease] = (
        inpatient_df[dx_cols]
            .apply(lambda col: col.str.contains(pattern, regex=True, na=False))
            .any(axis=1)
            .astype(int)
    )

In [91]:
disease_flags = (
    inpatient_df
        .groupby(["bene_id", "year"])[list(disease_map.keys())]
        .max()
        .reset_index()
)

In [92]:
disease_flags.head()

,bene_id,year,diabetes,ckd,chf,copd,cancer,stroke
0,-10000010254618,2015,1,0,0,0,0,0
1,-10000010254653,2015,0,0,0,0,1,0
2,-10000010254653,2017,0,0,0,0,1,0
3,-10000010254656,2017,0,0,0,0,0,0
4,-10000010254656,2018,0,0,0,0,0,0


In [93]:
disease_cols = list(disease_map.keys())

model_df["comorbidity_count"] = model_df[disease_cols].sum(axis=1)

In [94]:
model_df.describe()

,bene_id,year,total_inpatient_paid,total_inpatient_charge,total_deductible,total_coinsurance,alzheimers,chf,ckd,copd,diabetes,stroke,cancer,ischemic_hd,depression,osteoporosis,rheumatoid_oa,comorbidity_count
count,8.691700e+04,86917.000000,86917.0,86917.0,86917.0,86917.0,86917.000000,86917.000000,86917.000000,86917.000000,86917.000000,86917.000000,86917.000000,86917.000000,86917.000000,86917.000000,86917.000000,86917.000000
mean,-1.000001e+13,2020.522901,0.0,0.0,0.0,0.0,0.016142,0.001070,0.138765,0.013058,0.216264,0.000012,0.042673,0.221901,0.009756,0.060092,0.084667,0.411841
std,9.609701e+03,3.129600,0.0,0.0,0.0,0.0,0.126022,0.032693,0.345703,0.113526,0.411699,0.003392,0.202120,0.415528,0.098292,0.237658,0.278387,0.761309
min,-1.000001e+13,2015.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-1.000001e+13,2018.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-1.000001e+13,2021.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,-1.000001e+13,2023.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,-1.000001e+13,2025.000000,0.0,0.0,0.0,0.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4.000000


In [95]:
model_df.groupby("comorbidity_count")["total_inpatient_paid"].mean()

comorbidity_count
0.0    0.0
1.0    0.0
2.0    0.0
3.0    0.0
4.0    0.0
Name: total_inpatient_paid, dtype: float64

Debugging the comorbidity count and total ipatient paid calculations

In [96]:
inpatient_df["clm_pmt_amt"].describe()

count     58066.000000
mean      13638.307734
std       35993.907780
min          62.440000
25%         945.370000
50%        1481.715000
75%        9932.040000
max      598716.310000
Name: clm_pmt_amt, dtype: float64

In [97]:
inpatient_df["year"].value_counts(dropna=False)

year
2022    10598
2021     9249
2020     8775
2019     6791
2018     6297
2017     5515
2016     4928
2015     4050
2023     1863
Name: count, dtype: int64

In [98]:
annual_cost = (
    inpatient_df
        .groupby(["bene_id", "year"])["clm_pmt_amt"]
        .sum()
        .reset_index()
)

In [99]:
annual_cost = annual_cost.rename(columns={
    "clm_pmt_amt": "total_inpatient_paid"
})

In [100]:
annual_cost.describe()

,year,total_inpatient_paid
count,11545.000000,1.154500e+04
mean,2018.989259,6.859437e+04
std,2.343299,5.776398e+05
min,2015.000000,6.244000e+01
25%,2017.000000,1.650000e+02
50%,2019.000000,5.859360e+03
75%,2021.000000,2.388212e+04
max,2023.000000,2.572090e+07


In [101]:
model_df = annual_cost.merge(
    disease_flags,
    on=["bene_id", "year"],
    how="left"
).fillna(0)

In [102]:
disease_cols = list(disease_map.keys())

model_df["comorbidity_count"] = model_df[disease_cols].sum(axis=1)

In [103]:
model_df.groupby("comorbidity_count")["total_inpatient_paid"].mean()

comorbidity_count
0     34597.927688
1    107841.661110
2     69017.836351
3    432937.503442
4    177606.320000
Name: total_inpatient_paid, dtype: float64

In [106]:
model_df["high_cost_flag"] = (
    model_df
        .groupby("year")["total_inpatient_paid"]
        .transform(lambda x: x >= x.quantile(0.95))
        .astype(int)
)
model_df["high_cost_flag"].value_counts()

high_cost_flag
0    10964
1      581
Name: count, dtype: int64

In [107]:
model_df.sort_values("total_inpatient_paid", ascending=False).head(10)

,bene_id,year,total_inpatient_paid,diabetes,ckd,chf,copd,cancer,stroke,comorbidity_count,high_cost_flag
5977,-10000010271384,2020,25720895.06,0,0,0,0,1,0,1,1
3629,-10000010264641,2022,19454109.68,1,0,0,0,1,0,2,1
1945,-10000010259769,2021,18282120.06,1,1,0,0,1,0,3,1
3881,-10000010265392,2022,16785460.22,0,0,0,0,1,0,1,1
9243,-10000010280870,2022,15787070.98,0,0,0,0,1,0,1,1
5979,-10000010271384,2022,14061334.76,0,0,0,0,1,0,1,1
2045,-10000010260059,2022,13224979.60,0,0,0,0,1,0,1,1
9214,-10000010280747,2022,11567472.80,0,0,0,0,1,0,1,1
8350,-10000010277912,2021,10611675.74,1,1,0,0,1,0,3,1
8211,-10000010277596,2022,9377179.68,0,0,0,0,1,0,1,1


In [111]:
model_df.groupby("high_cost_flag")[disease_cols].mean()

,diabetes,ckd,chf,copd,cancer,stroke
high_cost_flag,,,,,,
0,0.455947,0.261675,0.001368,0.023805,0.071416,0.0
1,0.519793,0.302926,0.010327,0.039587,0.151463,0.0


In [121]:
feature_cols = [
    "diabetes",
    "ckd",
    "chf",
    "copd",
    "cancer",
    "stroke",
    "comorbidity_count"
]

X = model_df[feature_cols]
y = model_df["high_cost_flag"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

log_model.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [122]:
y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]

In [123]:
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.96      0.47      0.63      2193
           1       0.06      0.63      0.11       116

    accuracy                           0.48      2309
   macro avg       0.51      0.55      0.37      2309
weighted avg       0.91      0.48      0.61      2309

ROC-AUC: 0.5759057030992027


Η λογιστική παλινδρόμηση παρουσίασε περιορισμένη προβλεπτική ικανότητα (ROC-AUC = 0.576) στην αναγνώριση δικαιούχων υψηλού κόστους. Παρότι επιτεύχθηκε ικανοποιητική ευαισθησία (recall = 63%), η ακρίβεια (precision = 6%) ήταν ιδιαίτερα χαμηλή, γεγονός που υποδηλώνει υψηλό ποσοστό ψευδώς θετικών προβλέψεων. Τα αποτελέσματα υποδηλώνουν ότι οι δυαδικοί δείκτες νοσημάτων δεν επαρκούν για την ακριβή πρόβλεψη της υψηλής δαπάνης, επιβεβαιώνοντας τη σύνθετη και πολυπαραγοντική φύση του κόστους υγειονομικής περίθαλψης.